# 第 11 章：领域数据工程

这个 notebook 对应 `lessons/11_domain_data_engineering.md`，演示领域样本 schema、脱敏、重复/近重复检查、source split 泄漏检查和数据质量报告。

In [ ]:
import tempfile
from pathlib import Path

from src.data.domain_data import (
    DomainExample,
    assert_no_domain_split_leakage,
    build_quality_report,
    duplicate_count,
    find_near_duplicate_pairs,
    redact_sensitive_text,
    write_quality_report,
)
from src.data.text_datasets import ChatMessage

## 1. Domain Example Schema

领域样本需要记录 `id`、`source_id`、来源类型、创建方式、许可、风险标签和结构化 messages。

In [ ]:
def make_example(example_id, source_id, content, risk_tags):
    return DomainExample(
        id=example_id,
        source_id=source_id,
        source_type="contract_clause",
        created_by="manual",
        license="internal_review_only",
        contains_personal_data=False,
        risk_tags=risk_tags,
        messages=[
            ChatMessage(role="user", content=content),
            ChatMessage(role="assistant", content="需要人工复核"),
        ],
    )


examples = [
    make_example("a", "source_a", "合同电话13812345678", ["contract"]),
    make_example("b", "source_b", "医学危险信号需要就医", ["medical", "needs_human_review"]),
    make_example("c", "source_c", "合同条款要求甲方承担全部责任", ["contract"]),
    make_example("d", "source_d", "合同条款要求甲方承担全部责任。", ["contract"]),
]
print(examples[0])

## 2. 脱敏

脱敏保留任务结构，用占位符替换电话、身份证、地址、金额和日期等敏感字段。

In [ ]:
sensitive = (
    "张三电话13812345678，身份证110101199001011234，"
    "住北京市朝阳区。金额人民币20万元，日期2026年5月28日。"
)
redacted = redact_sensitive_text(sensitive)
print(redacted.text)
print(redacted.hits)

## 3. 重复与近重复

完全重复和近重复都可能造成 train/test 泄漏，尤其是合同条款和医学问答。

In [ ]:
print("duplicate_count:", duplicate_count(examples))
print("near_duplicate_pairs:", find_near_duplicate_pairs(examples, threshold=0.8))

## 4. Split 泄漏检查

同一个 `id` 或 `source_id` 不能同时进入 train / val / test。

In [ ]:
train = examples[:2]
val = examples[2:3]
test = examples[3:]
assert_no_domain_split_leakage(train, val, test)
print("split ok")

## 5. 数据质量报告

报告应包含样本数、长度分布、风险标签、重复率、脱敏命中和切分规则。

In [ ]:
report = build_quality_report(
    examples,
    split_rule="split by source_id",
    schema_error_count=0,
)

with tempfile.TemporaryDirectory() as tmpdir:
    path = Path(tmpdir) / "data_quality_report.md"
    write_quality_report(path, report)
    print(path.read_text())